<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Compare simulation assumptions and compute annualized NPV comparison outputs.
- **Primary inputs**: reporting/runs/*/policies/indicator.csv
- **Primary outputs**: npv_annual.csv and comparison figures
- **Refactor role**: Keep in reporting layer; use shared run registry instead of hardcoded simulation folders.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Map labels to run folders for model-assumption variants.
2. Load NPV-related indicators from each run.
3. Compare annual NPV metrics and export a consolidated table.

### Inputs
- `project/analysis/post_processing/reporting/runs/<run_id>/policies/indicator.csv` for each run in `dict_simulation`

### Outputs
- `npv_annual.csv` in notebook output folder.
- Assumption-comparison plot/table views.

### How To Reuse
- Update `dict_simulation` entries to the run ids you want to compare.


In [ ]:
import pandas as pd
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "project" / "utils.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find repository root containing project/utils.py")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from project.analysis.post_processing.shared.io_indicators import load_indicator
from project.analysis.post_processing.shared.paths import POST_PROCESSING_ROOT, resolve_run
from project.utils import slope_graph

REPORTING_ROOT = POST_PROCESSING_ROOT / "reporting"

In [ ]:
dict_simulation = {
    'Friction': 'policies_scenarios_20240527_120424',
    'Friction biased': 'policies_scenarios_20240527_120434',
    'No friction': 'policies_scenarios_20240527_120442',
    'No friction biased': 'policies_scenarios_20240527_120452'
}

available_runs = sorted(path.name for path in (REPORTING_ROOT / "runs").iterdir() if path.is_dir())
missing_runs = {
    label: run_id
    for label, run_id in dict_simulation.items()
    if not resolve_run("reporting", run_id).is_dir()
}

if missing_runs:
    missing_text = ", ".join(f"{label} -> {run_id}" for label, run_id in missing_runs.items())
    raise FileNotFoundError(
        f"Missing reporting runs: {missing_text}. Available runs: {available_runs}"
    )

folder_out = REPORTING_ROOT / 'output_comparison'
folder_out.mkdir(parents=True, exist_ok=True)

In [ ]:
dict_output = {}
for label, run_id in dict_simulation.items():
    run_folder = resolve_run("reporting", run_id)
    indicator_file = run_folder / "policies" / "indicator.csv"
    if not indicator_file.is_file():
        raise FileNotFoundError(f"Missing indicator file: {indicator_file}")
    dict_output[label] = load_indicator(run_folder)

dict_output = pd.concat(dict_output, axis=0).rename_axis(['Scenario', 'Indicator'], axis=0)

dict_replace = {'carbon_tax' : 'Carbon tax',
                'carbon_tax_high' : 'Carbon tax + EU-ETS 2',
                'cee_2018' : 'WCO 2018',
                'cee_2021' : 'WCO 2021',
                'cee_2024' : 'WCO 2024',
                'obligation' : 'Mandatory renovation',
                'restriction_gas' : 'Ban gas',
                'subsidies_2018' : 'Subsidies 2018',
                'subsidies_2021' : 'Subsidies 2021',
                'subsidies_2024' : 'Subsidies 2024',
                'zero_interest_loan' : 'Zero interest loan'}

order = ['Carbon tax', 'Carbon tax + EU-ETS 2', 'WCO 2018', 'WCO 2021', 'WCO 2024', 'Subsidies 2018', 'Subsidies 2021', 'Subsidies 2024', 'Zero interest loan', 'Mandatory renovation', 'Ban gas']
dict_output = dict_output.rename(columns=dict_replace)
dict_output = dict_output.T
dict_output = dict_output.loc[[i for i in order if i in dict_output.index], :]


### NPV annual
Assessing the welfare implication under different modelling assumptions.

In [ ]:
temp = dict_output.xs('NPV annual (Billion euro/year)', level='Indicator', axis=1)
temp.to_csv(folder_out / 'npv_annual.csv')
display(temp)

### Slope graph — welfare across normative perspectives

In [ ]:
# Rename columns to match paper labels (Table 6 order: left = most frictions, right = least)
perspective_order = ["Friction", "Friction biased", "No friction", "No friction biased"]
perspective_labels = {
    "Friction": "Friction",
    "Friction biased": "Friction\nbiased",
    "No friction": "No friction",
    "No friction biased": "No friction\nbiased",
}

# Policy colors consistent with the rest of the paper
color_dict = {
    "Carbon tax": "darkgreen",
    "Carbon tax + EU-ETS 2": "darkgreen",
    "WCO 2024": "darkorange",
    "Subsidies 2024": "royalblue",
    "Zero interest loan": "darkmagenta",
    "Mandatory renovation": "firebrick",
    "Rental ban": "firebrick",
    "Ban gas": "red",
}

slope_df = temp[[c for c in perspective_order if c in temp.columns]].copy()
slope_df = slope_df.rename(columns=perspective_labels)

from matplotlib.ticker import MaxNLocator, StrMethodFormatter

fig = slope_graph(
    slope_df,
    xlabel="Cost-benefit impact (Billion €/year)",
    color_dict=color_dict,
    figsize=(11, 8.5),
    annotation_gap=0.03,
)
ax = fig.axes[0]
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}"))
fig.suptitle(
    "Annual NPV by Policy Across Friction and Bias Assumptions",
    fontsize=16,
    fontweight="bold",
    color="dimgrey",
    y=0.99,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(folder_out / "slope_graph_perspectives.pdf", bbox_inches="tight", dpi=300)
fig

### Full friction

In [ ]:
dict_output